In [0]:
from pyspark.sql import functions as F

silver_maintenance = spark.table(
    "workspace.transportation_analytics.silver_maintenance_records"
)

silver_trucks = spark.table(
    "workspace.transportation_analytics.silver_trucks"
)

silver_maintenance.printSchema()
silver_trucks.printSchema()

In [0]:
silver_utilization = spark.table(
    "workspace.transportation_analytics.silver_truck_utilization_metrics"
)

gold_maintenance_analysis = (
    silver_maintenance
    .join(
        silver_utilization.select(
            "truck_id",
            "total_miles"
        ),
        on="truck_id",
        how="left"
    )
)

display(gold_maintenance_analysis)

In [0]:
truck_mileage = (
    silver_utilization
    .groupBy("truck_id")
    .agg(
        F.sum("total_miles").alias("total_miles")
    )
)

display(truck_mileage)

In [0]:
maintenance_by_truck = (
    silver_maintenance
    .groupBy("truck_id")
    .agg(
        F.count("maintenance_id").alias("maintenance_events"),
        F.sum("total_cost").alias("total_maintenance_cost"),
        F.sum("downtime_hours").alias("total_downtime_hours"),
        F.sum("labor_cost").alias("total_labor_cost"),
        F.sum("parts_cost").alias("total_parts_cost")
    )
)

display(maintenance_by_truck)

In [0]:
gold_maintenance_analysis = (
    maintenance_by_truck
    .join(
        truck_mileage,
        on="truck_id",
        how="left"
    )
)

display(gold_maintenance_analysis)

In [0]:
gold_maintenance_analysis = (
    gold_maintenance_analysis
    .withColumn(
        "maintenance_cost_per_mile",
        F.when(
            F.col("total_miles") > 0,
            F.round(
                F.col("total_maintenance_cost") / F.col("total_miles"),
                4
            )
        )
    )
)

display(
    gold_maintenance_analysis.select(
        "truck_id",
        "maintenance_events",
        "total_maintenance_cost",
        "total_downtime_hours",
        "total_miles",
        "maintenance_cost_per_mile"
    )
)

In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "workspace.transportation_analytics.gold_maintenance_analysis"
)

target.alias("t").merge(
    gold_maintenance_analysis.alias("s"),
    "t.truck_id = s.truck_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Gold Maintenance Analysis table updated using MERGE")

In [0]:
display(
    spark.table(
        "workspace.transportation_analytics.gold_maintenance_analysis"
    )
)

In [0]:
display(
    gold_maintenance_analysis
    .orderBy(F.desc("maintenance_cost_per_mile"))
    .select(
        "truck_id",
        "total_maintenance_cost",
        "total_miles",
        "maintenance_cost_per_mile"
    )
)

Databricks visualization. Run in Databricks to view.

In [0]:
display(
    gold_maintenance_analysis
    .orderBy(F.desc("total_downtime_hours"))
    .select(
        "truck_id",
        "total_downtime_hours",
        "total_maintenance_cost"
    )
)

Databricks visualization. Run in Databricks to view.